In [ ]:
class UNet3D(nn.Module):
    def __init__(self, in_channels, out_channels, base_channels, latent_dim=None):
        super(UNet3D, self).__init__()
        
        self.latent_dim = latent_dim
        self.rec_loss = nn.MSELoss() 
        self.time_loss = LogLikelihoodLossWithCensoring(dist_type='weibull')
        # Encoder and decoder layers
        self.enc1 = Encoder3D(in_channels, base_channels)
        self.enc2 = Encoder3D(base_channels, base_channels * 2)
        self.enc3 = Encoder3D(base_channels * 2, base_channels * 4)
        self.enc4 = Encoder3D(base_channels * 4, base_channels * 8)
        
        if latent_dim is not None:
            self.latent_space = LatentSpaceGaussian(latent_dim)
            # Calculate size for linear layer to output latent parameters
            self.latent_mu = nn.Linear(base_channels * 8 * 8 * 8 * 4, 2 * latent_dim)  # Assuming the output from the encoder is [4, 128, 8, 8, 4]

            #self.fc_decode = nn.Linear(latent_dim, base_channels * 8 * 8 * 4)
            self.fc_decode = nn.Linear(latent_dim, 128 * 8 * 8 * 4)  # Outputs channels to match the decoder's expectations

        
        self.dec4 = Decoder3D(base_channels * 8, base_channels * 4)
        self.dec3 = Decoder3D(base_channels * 4, base_channels * 2)
        self.dec2 = Decoder3D(base_channels * 2, base_channels)
        self.dec1 = Decoder3D(base_channels, out_channels)

        if latent_dim is not None:
            self.time_decoder = TimeDecoder(latent_dim)
    
    def forward(self, x):
        batch_size, _, H, W, D = x.shape 
        #print("Input shape:", x.shape)  # Print input shape, expect (4, 1, 128, 128, 64)

        enc1 = self.enc1(x)
        #print("After enc1 shape:", enc1.shape)  # Expect (4, base_channels, 64, 64, 32) after downsampling

        enc2 = self.enc2(enc1)
        #print("After enc2 shape:", enc2.shape)  # Expect (4, base_channels*2, 32, 32, 16)

        enc3 = self.enc3(enc2)
        #print("After enc3 shape:", enc3.shape)  # Expect (4, base_channels*4, 16, 16, 8)

        enc4 = self.enc4(enc3)
        #print("After enc4 shape:", enc4.shape)  # Expect (4, base_channels*8, 8, 8, 4)
        
        out_data = {}  # Dictionary to store outputs
        
        if self.latent_dim is not None:
            # Flatten and pass through latent layers
            enc4_flat = enc4.view(enc4.size(0), -1)
            #print("enc4_flat shape:", enc4_flat.shape)  # Expect (4, base_channels*8*8*8)

            # Get the latent parameters directly from the linear layer
            latent_params = self.latent_mu(enc4_flat)
            print("latent_params shape:", latent_params.shape)  # Expect (4, 2 * latent_dim)

            mu, log_var = self.latent_space.get_latent_params(latent_params)
            #print("mu shape:", mu.shape)  # Expect (4, latent_dim)
            #print("log_var shape:", log_var.shape)  # Expect (4, latent_dim)
            
            z = self.latent_space.sample_latent((mu, log_var))
            #print("z shape:", z.shape)  # Expect (4, latent_dim)
            
            # Decode the latent sample
            enc4_decoded = self.fc_decode(z)
            #print("enc4_decoded shape:", enc4_decoded.shape)  # Expect (4, base_channels * 8 * 8 * 8)

            enc4 = enc4_decoded.view(enc4_decoded.size(0), -1, 8, 8, 4)
            #print("enc4 reshaped shape:", enc4.shape)  # Expect (4, base_channels * 8, 8, 8, 4)
            
            # Predict Weibull parameters
            weibull_params = self.time_decoder(z)
            #print("weibull_params shape:", weibull_params.shape)  # Expect (4, 2) for Weibull params
            
            # Add to out_data dictionary
            out_data['latent_sample'] = z
            out_data['latent_params'] = latent_params
            out_data['time_params'] = weibull_params
            # out_data['mu'] = mu
            # out_data['log_var'] = log_var
        
        # Decoding path
        dec4 = self.dec4(enc4)
        #print("dec4 shape:", dec4.shape)  # Expect (4, base_channels*4, 16, 16, 8)

        dec3 = self.dec3(dec4 + enc3)
        #print("dec3 shape:", dec3.shape)  # Expect (4, base_channels*2, 32, 32, 16)

        dec2 = self.dec2(dec3 + enc2)
        #print("dec2 shape:", dec2.shape)  # Expect (4, base_channels, 64, 64, 32)

        dec1 = self.dec1(dec2 + enc1)
        #print("dec1 shape:", dec1.shape)  # Expect (4, out_channels, 128, 128, 64)
        
        # Reconstruction output
        out_data['reconstruction'] = dec1
        
        return out_data